# 01 — EDA & Data Contract
**Fase 1** · Sistem Analisis Sentimen Ulasan Gojek

Notebook ini **memanggil** `ml/src/eda.py`; ia tidak menduplikasi logikanya.
Dengan begitu angka di notebook dan angka yang dipakai fase berikutnya dijamin identik.

Sumber data: `data/raw/ulasan_com.gojek.app.csv` — unduhan langsung Kaggle
(`pandaa12`, versi 1), tanpa modifikasi. Lihat `docs/data_provenance_notes.md`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from ml.src.config import load_config
from ml.src import eda

pd.set_option('display.width', 120)
cfg = load_config()
print('seed:', cfg['seed'], '| skema label:', cfg['labeling']['scheme'])

seed: 42 | skema label: binary


## 1. Muat & validasi data

Validasi skema dilakukan di `ingest.py` lewat `assert` — bila sel ini lolos, kontrak data terpenuhi.

In [2]:
from ml.src.ingest import load_raw
df = eda.prepare(load_raw(cfg))
print(df.shape)
df.head(3)

(100000, 10)


,Nama User,Ulasan,Rating,Tanggal,Likes,Versi App,versi_minor,word_count,bulan,sentimen_aktual
0,Pengguna Google,Mohon agar Gojek menambahkan fitur pembayaran ...,5,2025-12-31 17:12:17,1,5.44.1,5.44,16,2025-12,1.0
1,Pengguna Google,aplikasi kek taik saya pesan gojek prioritas m...,1,2025-12-31 17:09:24,0,<NA>,<NA>,11,2025-12,0.0
2,Pengguna Google,oke pokonya,5,2025-12-31 16:58:17,0,5.44.1,5.44,2,2025-12,1.0


## 2. Profil dasar

> Menghasilkan `[A-01]`, `[A-02]`

In [3]:
eda.profil_dataset(df)

,properti,nilai
0,Jumlah baris,"100,000"
1,Jumlah kolom,10
2,Periode awal,2024-05-21
3,Periode akhir,2025-12-31
4,Jumlah bulan,20
5,SHA-256 berkas,9b7f8e50737f9f17bce4ab8a44f7d2f8587deea86c6ad1...
6,Missing · Nama User,0
7,Missing · Ulasan,0
8,Missing · Rating,0
9,Missing · Tanggal,0


## 3. Duplikasi

> Menghasilkan `[A-03]` — bahan Temuan 2

In [4]:
eda.duplikasi(df)

,jenis,jumlah,persen
0,Duplikat baris penuh,0,0.000
1,Duplikat teks Ulasan,32101,32.101


## 4. Distribusi rating → **Gambar 1**

In [5]:
eda.distribusi_rating(df)

,rating,jumlah,persen
0,1,23178,23.2
1,2,3554,3.6
2,3,3534,3.5
3,4,5083,5.1
4,5,64651,64.7


## 5. Asimetri panjang ulasan → **Gambar 2**

Ini temuan paling kritis (Temuan 2). Perbedaan panjang antar kelas menentukan
seluruh protokol evaluasi di Fase 3.

In [6]:
eda.panjang_per_rating(df)

,rating,rata_kata,median_kata,jumlah
0,1,20.8,15.0,23178
1,2,22.4,17.0,3554
2,3,20.0,15.0,3534
3,4,10.0,4.0,5083
4,5,4.4,2.0,64651


### 5b. Ulasan sangat pendek → **Gambar 3**

In [7]:
display(eda.ulasan_pendek(df))
pd.read_csv(ROOT / 'docs/tables/teks_paling_berulang.csv').head(10)

,kelompok,jumlah,persen
0,≤2 kata,39859,39.859
1,≥5 kata (informatif),45685,45.685


,teks,frekuensi
0,Mantap,2063
1,mantap,1911
2,ok,1740
3,Ok,1620
4,bagus,1403
5,Bagus,1233
6,Sangat membantu,1149
7,Good,1134
8,good,1092
9,sangat membantu,951


## 6. Kontras kelas negatif vs positif

Korpus negatif jauh lebih bersih — inilah alasan ekstraksi topik (Fase 5)
hanya dijalankan pada kelas negatif.

In [8]:
eda.kontras_kelas(df)

,kelas,jumlah,duplikat_teks,ulasan_le2kata,persen_duplikat,persen_le2kata
0,Negatif,26732,546,1614,2.0,6.0
1,Positif,69734,31214,37911,44.8,54.4


## 7. Kelayakan skema 3 kelas → dasar keputusan H-3

> Menghasilkan `[A-23]` (baseline mayoritas) dan `[A-24]` (rasio kelas netral)

In [9]:
eda.komposisi_kelas(df)

,skema,kelas,jumlah,persen
0,3 kelas,Negatif,26732,26.73
1,3 kelas,Netral,3534,3.53
2,3 kelas,Positif,69734,69.73
3,Biner,Negatif,26732,27.71
4,Biner,Positif,69734,72.29


## 8. Analisis versi aplikasi → **Gambar 5**

In [10]:
eda.cakupan_versi(df)

,ambang,jumlah_versi,ulasan_tercakup,persen_cakupan_berversi
0,30,86,76213,97.6
1,100,66,75247,96.4
2,200,64,74891,95.9
3,500,56,72166,92.4


## 9. Profil kolom Likes

Distribusi sangat timpang — dipakai sebagai filter dashboard, bukan fitur model.

In [11]:
eda.profil_likes(df)

,statistik,nilai
0,count,100000.000000
1,mean,1.059370
2,std,19.033131
3,min,0.000000
4,25%,0.000000
5,50%,0.000000
6,75%,0.000000
7,max,2904.000000
8,persen_likes_nol,83.719000


## 10. Volume temporal → **Gambar 4**

In [12]:
eda.volume_bulanan(df)

,bulan,jumlah,rata_rating,periode_parsial
0,2024-05,1879,3.97,True
1,2024-06,6275,3.93,False
2,2024-07,6641,4.03,False
3,2024-08,6186,4.08,False
4,2024-09,6362,3.98,False
5,2024-10,5453,3.83,False
6,2024-11,5622,3.61,False
7,2024-12,6571,3.67,False
8,2025-01,5265,3.77,False
9,2025-02,4777,3.89,False


## 11. Tabel rujukan angka

`angka_storytelling.csv` berisi kolom `kalimat_siap_tempel` — dipakai saat
menulis naskah (di luar repo, per H-4) agar tidak ada angka yang ditulis dari ingatan.

In [13]:
angka = pd.read_csv(ROOT / 'docs/tables/angka_storytelling.csv')
for r in angka.itertuples():
    print(f"[{r.label}] {r.kalimat_siap_tempel}")

[A-01] 100,000 ulasan
[A-02] 21,910 baris (21.9%) tidak mencantumkan versi aplikasi
[A-03] 32,101 ulasan (32.1%) merupakan duplikat teks
[A-11] bintang 1: 23,178 ulasan (23.2%)
[A-12] bintang 2: 3,554 ulasan (3.6%)
[A-13] bintang 3: 3,534 ulasan (3.5%)
[A-14] bintang 4: 5,083 ulasan (5.1%)
[A-15] bintang 5: 64,651 ulasan (64.7%)
[A-20] ulasan bintang 5 rata-rata hanya 4.4 kata, sementara bintang 1 mencapai 20.8 kata
[A-21] 39,859 ulasan (39.9%) hanya berisi dua kata atau kurang
[A-22] pada kelas negatif hanya 2.0% ulasan yang merupakan duplikat teks, dibanding 44.8% pada kelas positif
[A-23] baseline mayoritas pada skema biner adalah 72.3% — angka yang harus dikalahkan model
[A-24] kelas netral hanya 3,534 ulasan (3.5%), rasio 1:19 terhadap kelas positif
[A-30] terdapat 291 versi aplikasi unik
[A-40] 83.7% ulasan tidak pernah mendapat satu pun like
